# Somo la 13 - Kumbukumbu ya Wakala na Miundo ya Maarifa ya Cognee


## Usanidi

Daftari hili linaonyesha jinsi ya kujenga **msaidizi wa kufunga msimbo** mwenye akili kwa kutumia kumbukumbu endelevu kwa kutumia michoro ya maarifa ya [**Cognee**](https://www.cognee.ai/) na **Microsoft Agent Framework** (MAF).

Cognee hubadilisha maandishi yasiyopangwa kuwa mchoro wa maarifa uliopangwa na unaoweza kuulizwa unaotegemea vector embeddings — ukimpa wakala wako kumbukumbu ya muda mrefu yenye uhusiano tajiri.

### Utajifunza Nini
1. **Jenga Michoro ya Maarifa**: Badilisha maelezo ya waendelezaji na mbinu bora kuwa maarifa yaliyoandikwa na yanayoweza kuulizwa.
2. **Unganisha Cognee na MAF**: Tumia kazi za `@tool` kuruhusu wakala wa MAF kuuliza mchoro wa maarifa wa Cognee.
3. **Mazungumzo Yanayojua Kikao**: Hifadhi muktadha kwa maswali mengi katika kikao kimoja.
4. **Kumbukumbu ya Muda Mrefu**: Hifadhi maarifa muhimu kwenye vikao na kuyapata tena katika mazungumzo mapya.

### Sharti la Kuwa Tayari
- Python 3.9+
- Redis inayoendesha kwa ndani (`docker run -d -p 6379:6379 redis`) kwa usimamizi wa kikao
- Kitufe cha API cha LLM (k.m. OpenAI) — weka `LLM_API_KEY` katika `.env`
- `CACHING=true` katika `.env` (inahitajika kwa vikao vya Cognee)
- Mradi wa Microsoft Foundry na mfano wa mazungumzo uliowekwa
- `AZURE_AI_PROJECT_ENDPOINT` na `AZURE_AI_MODEL_DEPLOYMENT_NAME` katika `.env`
- Azure CLI imeingia kwa usahihi (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## Aina za Kumbukumbu za Wakala

Daftari hili linachunguza aina tatu za kumbukumbu kutoka kwenye daftari kuu la Somo la 13, lakini linatumia Cognee kama nyenzo ya kumbukumbu ya muda mrefu:

| Aina ya Kumbukumbu | Mbinu | Muda wa Kuishi |
|---|---|---|
| **Inayotumika** | `agent.create_session()` (MAF) | Mazungumzo moja |
| **Muda mfupi** | Hifadhi ya kikao cha Cognee (Redis) | Kikao kimoja |
| **Muda mrefu** | Grafu ya maarifa ya Cognee + vekta | Kudumu |

### Miundo ya Kumbukumbu ya Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Andaa Hifadhi ya Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Sehemu ya 1 — Kujenga Hifadhidata ya Maarifa

Tunakusanya aina tatu za data ili kuunda hifadhidata ya maarifa ya kina kwa msaidizi wetu wa kufunga programu:

1. **Wasifu wa Mendelezaji** — ujuzi binafsi na msingi wa kiufundi
2. **Mbinu Bora za Python** — Zen ya Python na miongozo ya vitendo
3. **Mijadala ya Zamani** — vikao vya zamani vya maswali na majibu kati ya mendelezaji na wasaidizi wa AI


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Onyesha Mchoro wa Maarifa

Cognee inaweza kuonyesha mchoro wa HTML unaoingiliana wa vitu na mahusiano ambavyo ilivitambua.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Boreshaji Kumbukumbu na Memify

`memify()` inachambua grafu ya maarifa na kuunda sheria za werevu — ikitambua mifumo, mbinu bora, na uhusiano kati ya dhana.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Sehemu ya 2 — Wakala wa MAF na Zana za Cognee

Sasa tunaunda wakala wa MAF ambaye anaweza kuuliza grafu ya maarifa ya Cognee kupitia kazi za `@tool`. Hii inamruhusu wakala kutumia nguvu kamili ya utafutaji wa semantiki unaojua grafu huku akiendeleza muktadha wa mazungumzo kupitia vikao.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## Kumbukumbu ya Kazi na Vikao

`AgentSession` (iliyoundwa kupitia `agent.create_session()`) hutoa kumbukumbu ya kazi ndani ya kikao. Wakala anaweza kurejea nyuma kwa ujumbe wa awali huku pia akiulizia mchoro wa maarifa ya muda mrefu wa Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Kikao Kipya — Kumbukumbu ya Muda Mrefu Inadumu

Kuanzisha kikao kipya kunafuta kumbukumbu ya kazi, lakini grafu ya maarifa ya Cognee bado inapatikana. Wakala anaweza kupata maarifa yale yale ya muda mrefu katika mazungumzo mapya kabisa.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Muhtasari

Katika daftari hili umeunda msaidizi wa uandishi wa programu anayechanganya **kumbukumbu ya kazi ya MAF** (`agent.create_session()`) na **grafu ya maarifa ya muda mrefu ya Cognee**.

### Umejifunza Nini
1. **Ujenzi wa grafu ya maarifa**: Cognee hutumia maandishi yasiyo na muundo na kujenga grafu + kumbukumbu ya vekta.
2. **Kuongeza maarifa kwenye grafu kwa memify**: Ukweli unaotokana na grafu na uhusiano tajiri zaidi juu ya grafu uliopo.
3. **Muunganisho wa MAF + Cognee**: Kazi za `@tool` huruhusu wakala wa MAF kuuliza grafu ya Cognee kwa njia ya asili.
4. **Kumbukumbu ya kazi + kumbukumbu ya muda mrefu**: `AgentSession` (kupitia `agent.create_session()`) hutoa muktadha wa kikao wakati Cognee hutoa maarifa ya kudumu.
5. **Utafutaji uliosafishwa kwa NodeSets**: Lengo sehemu maalum za grafu ya maarifa (k.m. kanuni tu).

### Muhimu wa Kufahamu
- **Cognee** hubadilisha maandishi ghafi kuwa kumbukumbu iliyo na muundo na uhusiano — yenye nguvu zaidi kuliko hifadhi rahisi ya vekta.
- **Kazi za `@tool`** huunganisha mawakala wa MAF na mifumo ya maarifa ya nje kwa usafi.
- **`AgentSession`** (kupitia `agent.create_session()`) huwajengea muktadha wa mazungumzo tofauti na maarifa ya muda mrefu.
- Grafu moja la maarifa hutoa huduma kwa vikao na mawakala wengi.

### Matumizi ya Uhalisia
- **Msaidizi wa watengenezaji wa programu**: Mapitio ya msimbo, uchambuzi wa matukio, wasaidizi wa usanifu
- **Wasaidizi wanaotumia mteja**: Wakala wa msaada juu ya nyaraka za bidhaa, maswali yanayoulizwa mara kwa mara, na vidokezo vya CRM
- **Wasaidizi wa wataalamu wa ndani**: Msaada wa sera, sheria, au usalama anayefikiri juu ya miongozo
- **Tabaka moja la data zilizounganishwa**: Changanya data zilizo na muundo na zisizo na muundo kuwa grafu moja inayoweza kuulizwa

### Hatua Zifuatayo
- Jaribu ufahamu wa muda katika Cognee
- Tambua ontolojia ya OWL kwa ubora wa grafu maalum wa eneo
- Ongeza michakato ya mrejesho wa mtumiaji ili kuboresha upatikanaji kwa muda
- Panua kwa mifumo ya wakala wengi zinazoshirikisha tabaka moja la kumbukumbu la Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
